# Manual Error Analysis Notebook

Follow the eval flywheel methodology:
1. Load your Claude Code history
2. Review sessions one by one
3. Write open-ended notes ("journaling")
4. Let patterns emerge naturally
5. Build judges AFTER you understand your failure modes

In [ ]:
import sys
sys.path.insert(0, '..')

from analysis.history_analyzer import load_history, filter_by_date, generate_report
from analysis.golden_dataset import extract_sessions, session_summary

# Load your history
entries = load_history()
print(f'Total prompts: {len(entries)}')

# Optional: filter to a date range
# entries = filter_by_date(entries, start='2025-01-01', end='2025-12-31')

In [ ]:
# Quick overview
print(generate_report(entries))

In [ ]:
# Split into sessions (30 min gap = new session)
sessions = extract_sessions(entries, gap_minutes=30)
print(f'Found {len(sessions)} sessions')

# Show session summaries
for i, session in enumerate(sessions[-10:]):
    s = session_summary(session)
    print(f"\nSession {i+1}: {s['project']} ({s['prompt_count']} prompts, {s['duration_min']:.0f} min)")
    for prompt in s['first_prompts'][:2]:
        print(f"  > {prompt[:100]}")

## Step 1: Manual Review

Pick a session and review it carefully. For each session, write:
- **Goal**: What was the user trying to do?
- **Outcome**: Did it work?
- **Verdict**: PASS or FAIL
- **Critique**: Open-ended notes about what went right or wrong

Don't categorize yet — just journal.

In [ ]:
# Review a specific session in detail
session_idx = -1  # Change this to review different sessions
session = sessions[session_idx]

print(f"Session: {session_summary(session)['project']}")
print(f"Prompts: {len(session)}")
print(f"Duration: {session_summary(session)['duration_min']:.0f} min")
print('\n---\n')

for i, p in enumerate(session):
    text = p.get('display', '')[:300]
    print(f'[{i+1}] {text}')
    print()

## Step 2: Run Judges

After manual review, run the automated judges to compare.

In [ ]:
from judges import judge_secret_hygiene, judge_session_discipline, judge_prompt_efficiency, judge_topic_coherence

session = sessions[-1]  # Pick a session

print('Secret Hygiene:')
for p in session[:5]:
    v = judge_secret_hygiene(p.get('display', ''))
    if v['verdict'] == 'FAIL':
        print(f"  FAIL: {v['reason']}")

print(f'\nSession Discipline: {judge_session_discipline(session)}')
print(f'\nTopic Coherence: {judge_topic_coherence(session)}')

## Step 3: Export Golden Dataset

Generate a CSV of sessions for systematic annotation.

In [ ]:
from analysis.golden_dataset import export_for_annotation

# Export the 50 most recent sessions
path = export_for_annotation(sessions[-50:], output_path='golden_dataset.csv')
print(f'Annotation CSV written to: {path}')
print('Open it, review each session, fill in verdict + critique columns.')